In [44]:
%pip install datasets


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [45]:
import mlx.core as mx
import mlx.nn as nn

In [46]:
class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, maxlen: int, vocab_size:int, embed_dim: int):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(maxlen, embed_dim)

    def __call__(self, x):
        seq_len = x.shape[1]
        positions = mx.arange(seq_len)[None, :]
        return self.token_emb(x) + self.pos_emb(positions) 

In [47]:
class TransformerBlock(nn.Module):
    def __init__(self, emed_dim: int, num_heads:int, ff_dim: int):
        super().__init__()
        self.attention = nn.MultiHeadAttention(emed_dim, num_heads)
    def __call__(self, x, mask=None):
        attn_out = self.attention(x, x, x, mask=mask)
        x = x + attn_out
        return x

In [48]:

class NanoLLM(nn.Module):
    def __init__(self, maxlen: int, vocab_size:int, embed_dim:int, num_heads:int, feed_forward_dim:int , num_transformer_blocks: int):
        super().__init__()
        self.maxlen = maxlen
        self.embedding = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
        self.transformer_blocks = [
            TransformerBlock(embed_dim, num_heads, feed_forward_dim)
            for _ in range(num_transformer_blocks)
        ]
        self.output_layer = nn.Linear(embed_dim, vocab_size, bias=False)

    def __call__(self, token_ids):
        seq_len = token_ids.shape[1]

        mask = nn.MultiHeadAttention.create_additive_causal_mask(seq_len)

        x = self.embedding(token_ids)
        for block in self.transformer_blocks:
            x = block(x, mask=mask)

        logits = self.output_layer(x)
        return logits



    

In [49]:
from datasets import load_dataset
import os

def download_tinystories(output_file="TinyStories-100k.txt", num_stories=100000):
    print(f"Downloading TinyStories from Hugging Face...")
    # Load the training split
    dataset = load_dataset("roneneldan/TinyStories", split="train")
    
    # Slice exactly the number of stories you want
    print(f"Selecting {num_stories} stories...")
    subset = dataset.select(range(num_stories))
    
    # Write to a local text file
    print(f"Formatting and writing to {output_file}...")
    with open(output_file, "w", encoding="utf-8") as f:
        for item in subset:
            story = item['text'].strip()
            if story:
                # Append the EOS token between stories
                f.write(story + "\n<|endoftext|>\n")
                
    file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f"Done! Dataset saved to {output_file} ({file_size_mb:.2f} MB)")

# Run the download step
download_tinystories(output_file="TinyStories-100k.txt", num_stories=100000)

Selecting 100000 stories...
Formatting and writing to TinyStories-100k.txt...
Done! Dataset saved to TinyStories-100k.txt (85.77 MB)


In [50]:
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
import numpy as np
import tiktoken

def get_batches(file_path, tokenizer, batch_size=32, maxlen=128):
    """A lightweight generator to replace the Grain dataloader."""
    with open(file_path, 'r', encoding='utf-8') as f:
        data = f.read()
    
    # Split by the end token just like the JAX setup
    stories = data.split('<|endoftext|>')
    # Remove empty strings
    stories = [s.strip() + '<|endoftext|>' for s in stories if s.strip()]
    
    # Shuffle the dataset
    np.random.shuffle(stories)
    
    batch_x, batch_y = [], []
    
    for story in stories:
        tokens = tokenizer.encode(story, allowed_special={'<|endoftext|>'})
        
        # We need maxlen + 1 tokens because we shift the targets by 1
        # Example: if maxlen is 128, we need 129 tokens to get 128 inputs and 128 targets
        if len(tokens) > maxlen + 1:
            tokens = tokens[:maxlen + 1]
        
        # Pad with 0s if it's too short
        if len(tokens) < maxlen + 1:
            tokens.extend([0] * (maxlen + 1 - len(tokens)))
            
        # x is the input sequence, y is the target (shifted by 1)
        batch_x.append(tokens[:-1])
        batch_y.append(tokens[1:])
        
        if len(batch_x) == batch_size:
            # Yield MLX arrays directly
            yield mx.array(batch_x), mx.array(batch_y)
            batch_x, batch_y = [], []

In [51]:
# 1. Initialize Tokenizer and Model
tokenizer = tiktoken.get_encoding("gpt2")
model = NanoLLM(
    maxlen=128, 
    vocab_size=tokenizer.n_vocab, 
    embed_dim=192, 
    num_heads=6, 
    feed_forward_dim=512, 
    num_transformer_blocks=6
)


In [52]:
# 1. Initialize Tokenizer and Model
tokenizer = tiktoken.get_encoding("gpt2")
model = NanoLLM(
    maxlen=128, 
    vocab_size=tokenizer.n_vocab, 
    embed_dim=192, 
    num_heads=6, 
    feed_forward_dim=512, 
    num_transformer_blocks=6
)

# 2. Setup Optimizer (MLX equivalent of optax.adamw)
# Note: For simplicity, we are using a static learning rate here, 
# but MLX supports lr schedules via optim.step_decay etc.
optimizer = optim.AdamW(learning_rate=3e-4)

# 3. Define the Loss Function
def loss_fn(model, x, y):
    logits = model(x)
    # MLX has a built-in cross entropy loss that expects (logits, targets)
    loss = nn.losses.cross_entropy(logits, y)
    return mx.mean(loss)

# 4. Compile the Value and Grad function
# This is the exact MLX equivalent to nnx.value_and_grad
loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

# 5. Run the Loop
epochs = 5
step = 0

for epoch in range(epochs):
    # Initialize our generator
    dataloader = get_batches("TinyStories-100k.txt", tokenizer, batch_size=32, maxlen=128)
    
    for x, y in dataloader:
        # Calculate loss and gradients
        loss, grads = loss_and_grad_fn(model, x, y)
        
        # Apply gradients
        optimizer.update(model, grads)
        
        # CRITICAL: Force MLX to actually compute the updates on the Apple GPU
        mx.eval(model.parameters(), optimizer.state, loss)
        
        if step % 50 == 0:
            print(f"Epoch: {epoch + 1} | Step: {step} | Loss: {loss.item():.4f}")
            
        step += 1

Epoch: 1 | Step: 0 | Loss: 10.8266
Epoch: 1 | Step: 50 | Loss: 5.9461
Epoch: 1 | Step: 100 | Loss: 5.6107
Epoch: 1 | Step: 150 | Loss: 5.2950
Epoch: 1 | Step: 200 | Loss: 4.9307
Epoch: 1 | Step: 250 | Loss: 4.5847
Epoch: 1 | Step: 300 | Loss: 4.2845
Epoch: 1 | Step: 350 | Loss: 4.3064
Epoch: 1 | Step: 400 | Loss: 4.0243
Epoch: 1 | Step: 450 | Loss: 3.8484
Epoch: 1 | Step: 500 | Loss: 3.7691
Epoch: 1 | Step: 550 | Loss: 3.8548
Epoch: 1 | Step: 600 | Loss: 3.9880
Epoch: 1 | Step: 650 | Loss: 3.8109
Epoch: 1 | Step: 700 | Loss: 3.8814
Epoch: 1 | Step: 750 | Loss: 3.6469
Epoch: 1 | Step: 800 | Loss: 3.6962
Epoch: 1 | Step: 850 | Loss: 3.5966
Epoch: 1 | Step: 900 | Loss: 3.5697
Epoch: 1 | Step: 950 | Loss: 3.6199
Epoch: 1 | Step: 1000 | Loss: 3.7395
Epoch: 1 | Step: 1050 | Loss: 3.5999
Epoch: 1 | Step: 1100 | Loss: 3.6092
Epoch: 1 | Step: 1150 | Loss: 3.6909
Epoch: 1 | Step: 1200 | Loss: 3.4631
Epoch: 1 | Step: 1250 | Loss: 3.6353
Epoch: 1 | Step: 1300 | Loss: 3.4745
Epoch: 1 | Step: 1350 |

In [53]:
model.save_weights("small_checkpoint.safetensors")

print("Model saved successfully!")

Model saved successfully!


In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0):
    # Encode the prompt and convert to an MLX array with a batch dimension
    tokens = tokenizer.encode(prompt)
    x = mx.array(tokens)[None, :] # Shape: (1, seq_len)
    
    end_token_id = tokenizer.encode('<|endoftext|>', allowed_special={'<|endoftext|>'})[0]
    
    print(prompt, end="", flush=True)
    
    for _ in range(max_new_tokens):
        # Truncate context if it exceeds the model's max length
        if x.shape[1] > model.maxlen:
            x = x[:, -model.maxlen:]
            
        # Forward pass
        logits = model(x)
        
        # Grab the logits for the very last token in the sequence
        next_token_logits = logits[0, -1, :] / temperature
        
        # Get the highest probability token (greedy decoding)
        # Note: You can replace mx.argmax with mx.random.categorical for more creative outputs
        next_token = mx.argmax(next_token_logits, axis=-1).item()
        
        if next_token == end_token_id:
            break
            
        # Decode and print the single new token immediately
        word = tokenizer.decode([next_token])
        print(word, end="", flush=True)
        
        # Append the new token to our input sequence for the next loop
        x = mx.concatenate([x, mx.array([[next_token]])], axis=1)
        
    print("\n")

# Run inference
generate_text(model, tokenizer, "The king of jungle", max_new_tokens=1024, temperature=0.4)

There lived kid named Tom and was a very good day. He had a big box of toys and he had a lot of toys. He was very happy and he wanted to play with his toys.

One day, Tom was playing in the park. He saw a big tree with a big tree. He wanted to climb it, but he was too small. He tried to climb the tree, but he could not reach it.

Tom was sad. He wanted to help, but he was not sure. He tried to climb the tree, but he was too high. He could not reach the tree. He was sad.

Spot was sad. He tried to make his friends. He says he could not give him to the other animals.



In [56]:
# 1. Initialize the empty architecture
model = NanoLLM(
    maxlen=128, 
    vocab_size=tokenizer.n_vocab, 
    embed_dim=192, 
    num_heads=6, 
    feed_forward_dim=512, 
    num_transformer_blocks=6
)

# 2. Load the trained weights directly into the layers
model.load_weights("nanollm_tinystories.safetensors")

print("Weights loaded and ready for generation!")

Weights loaded and ready for generation!
